# 🛒 Bronze to Silver - Sales Orders Transformation

## 🎯 Objetivo
Transformar a tabela **FATO** de pedidos de vendas da camada Bronze para Silver, aplicando limpeza, conversões de tipo e validações.

## 📊 Tabelas
- **Origem**: `retail_dev.bronze.bronze_sales_orders`
- **Destino**: `retail_dev.silver.sales_orders`
- **Tipo**: **TABELA FATO** (eventos de vendas)

## 🔧 Transformações Aplicadas
1. **Conversão de tipos**: customer_id (STRING→INT), order_datetime (STRING→TIMESTAMP), number_of_line_items (STRING→INT)
2. **Limpeza**: TRIM + UPPER em customer_name
3. **Validação de FK**: customer_id existe em silver.customers
4. **Remoção de duplicados**: Por order_number (PK)
5. **Validação de arrays**: ordered_products não vazio, estruturas nested íntegras
6. **Auditoria**: data_quality_score, processed_at, source_table, pipeline_run_id
7. **Padronização**: Todos nomes de colunas em UPPER

## 📝 Características da Tabela
- **PK**: order_number (LONG)
- **FK**: customer_id → silver.customers.CUSTOMER_ID
- **Arrays nested**: clicked_items, ordered_products, promo_info
- **Structs**: ordered_products contém structs com promotion_info nested
- **SEM SCD Type 2**: Tabelas fato são append-only (eventos imutáveis)

## ⚠️ Problemas Identificados
- **74 duplicados** por order_number (4.074 registros, 4.000 únicos)
- **order_datetime**: STRING com Unix timestamp → converter para TIMESTAMP
- **customer_id**: STRING → INT para FK
- **number_of_line_items**: STRING → INT

In [0]:
from pyspark.sql import functions as F, Window
import uuid
from datetime import datetime

# Configurações
CATALOG = "retail_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
BRONZE_TABLE = "bronze_sales_orders"
SILVER_TABLE = "sales_orders"

# Gerar ID único para este pipeline run
pipeline_run_id = str(uuid.uuid4())
processing_timestamp = datetime.now()

print(f"🔧 Configuração carregada:")
print(f"   📦 Bronze: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
print(f"   ✨ Silver: {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")
print(f"   🆔 Pipeline Run ID: {pipeline_run_id}")
print(f"   ⏰ Timestamp: {processing_timestamp}")

## 📥 Leitura da Camada Bronze

Carregando dados da tabela `bronze_sales_orders` (tabela FATO).

In [0]:
bronze_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"

print(f"📖 Lendo tabela: {bronze_table_name}")
bronze_df = spark.table(bronze_table_name)

initial_count = bronze_df.count()

print(f"\n📊 Total de registros: {initial_count:,}")
print(f"\n📋 Schema da tabela Bronze:")
bronze_df.printSchema()

print(f"\n🔍 Amostra dos dados (3 primeiras linhas):")
display(bronze_df.limit(3))

# Análise rápida dos arrays
print(f"\n🔍 Análise dos arrays:")
print(f"   clicked_items type: {bronze_df.schema['clicked_items'].dataType}")
print(f"   ordered_products type: {bronze_df.schema['ordered_products'].dataType}")
print(f"   promo_info type: {bronze_df.schema['promo_info'].dataType}")

## 🔍 Auditoria Pré-Transformação

Analisando qualidade dos dados brutos:
- Duplicados por order_number (PK)
- Valores nulos
- Arrays vazios
- Validação de customer_id vs customers

In [0]:
print("🔍 AUDITORIA PRÉ-TRANSFORMAÇÃO")
print("=" * 80)

# 1. Duplicados por order_number (PK)
print("\n📊 1. Verificando duplicados (order_number):")
duplicates_check = bronze_df.groupBy("order_number").count().filter(F.col("count") > 1)
duplicates_count = duplicates_check.count()
total_duplicate_records = duplicates_check.agg(F.sum("count")).collect()[0][0] if duplicates_count > 0 else 0

print(f"   ⚠️ Total de order_numbers duplicados: {duplicates_count:,}")
print(f"   ⚠️ Total de registros duplicados: {total_duplicate_records:,}")
if duplicates_count > 0:
    print(f"\n   📋 Exemplos de duplicados:")
    display(duplicates_check.limit(10))

# 2. Valores nulos por coluna
print("\n📊 2. Valores nulos por coluna:")
for col in bronze_df.columns:
    null_count = bronze_df.filter(F.col(col).isNull()).count()
    null_pct = (null_count / initial_count) * 100 if initial_count > 0 else 0
    if null_count > 0:
        print(f"   ⚠️ {col}: {null_count:,} ({null_pct:.2f}%)")
    else:
        print(f"   ✅ {col}: 0 (0.00%)")

# 3. Análise de arrays vazios
print("\n📊 3. Análise de arrays:")
empty_clicked = bronze_df.filter(F.size(F.col("clicked_items")) == 0).count()
empty_ordered = bronze_df.filter(F.size(F.col("ordered_products")) == 0).count()
empty_promo = bronze_df.filter(F.size(F.col("promo_info")) == 0).count()

print(f"   clicked_items vazios: {empty_clicked:,} ({(empty_clicked/initial_count)*100:.2f}%)")
print(f"   ordered_products vazios: {empty_ordered:,} ({(empty_ordered/initial_count)*100:.2f}%)")
print(f"   promo_info vazios: {empty_promo:,} ({(empty_promo/initial_count)*100:.2f}%)")

# 4. Análise de tipos de dados
print("\n📊 4. Análise de tipos (problemas):")
print(f"   customer_id type: {bronze_df.schema['customer_id'].dataType} → precisa ser INT")
print(f"   order_datetime type: {bronze_df.schema['order_datetime'].dataType} → precisa ser TIMESTAMP")
print(f"   number_of_line_items type: {bronze_df.schema['number_of_line_items'].dataType} → precisa ser INT")

# 5. Análise de datas (COM PROTEÇÃO PARA STRINGS VAZIAS)
print("\n📊 5. Análise de order_datetime (STRING com Unix timestamp):")
# Validar se é numérico antes de fazer cast (mesma abordagem da célula 8)
date_sample = bronze_df.select(
    "order_datetime",
    F.when(
        F.col("order_datetime").rlike("^[0-9]+$"),
        F.from_unixtime(F.col("order_datetime").cast("long"))
    ).otherwise(None).alias("converted_date")
).filter(F.col("order_datetime").isNotNull()).limit(5)

print("   📋 Amostra de conversões (5 primeiras):")
display(date_sample)

# Verificar quantos order_datetime são vazios ou inválidos
empty_or_invalid_datetime = bronze_df.filter(
    (F.col("order_datetime").isNull()) | (~F.col("order_datetime").rlike("^[0-9]+$"))
).count()
valid_datetime = initial_count - empty_or_invalid_datetime

print(f"\n   ✅ order_datetime válidos (numéricos): {valid_datetime:,} ({(valid_datetime/initial_count)*100:.2f}%)")
print(f"   ⚠️ order_datetime vazios/inválidos: {empty_or_invalid_datetime:,} ({(empty_or_invalid_datetime/initial_count)*100:.2f}%)")

## 🧹 Limpeza e Padronização

Aplicando transformações:
1. **customer_id**: STRING → INT (FK para customers)
2. **customer_name**: TRIM + UPPER
3. **order_datetime**: STRING (Unix timestamp) → TIMESTAMP
4. **number_of_line_items**: STRING → INT
5. **Arrays**: Manter estrutura nested (clicked_items, ordered_products, promo_info)
6. **order_number**: LONG (PK - manter como está)

In [0]:
print("🧹 Aplicando transformações...")

transformed_df = bronze_df.select(
    # --- ARRAYS NESTED (manter estrutura) ---
    F.col("clicked_items"),  # ARRAY<ARRAY<STRING>>
    
    # --- IDENTIFICAÇÃO E FK ---
    # customer_id: STRING → INT (FK para silver.customers)
    # Converter apenas se for numérico válido, caso contrário NULL
    F.when(
        F.col("customer_id").rlike("^[0-9]+$"),
        F.col("customer_id").cast("int")
    ).otherwise(None).alias("customer_id"),
    
    # customer_name: TRIM + UPPER
    F.trim(F.upper(F.col("customer_name"))).alias("customer_name"),
    
    # --- MÉTRICAS ---
    # number_of_line_items: STRING → INT
    F.when(
        F.col("number_of_line_items").rlike("^[0-9]+$"),
        F.col("number_of_line_items").cast("int")
    ).otherwise(None).alias("number_of_line_items"),
    
    # --- TIMESTAMP ---
    # order_datetime: STRING (Unix timestamp) → TIMESTAMP
    # Converter apenas se for numérico válido
    F.when(
        F.col("order_datetime").rlike("^[0-9]+$"),
        F.from_unixtime(F.col("order_datetime").cast("long")).cast("timestamp")
    ).otherwise(None).alias("order_datetime"),
    
    # --- PK ---
    # order_number: LONG (manter como está)
    F.col("order_number"),
    
    # --- ARRAYS NESTED DE PRODUTOS E PROMOÇÕES (manter estrutura) ---
    F.col("ordered_products"),  # ARRAY<STRUCT>
    F.col("promo_info")  # ARRAY<STRUCT>
)

print(f"✅ Transformações aplicadas!")
print(f"\n📊 Total de registros após transformação: {transformed_df.count():,}")
print(f"\n📋 Schema após transformação:")
transformed_df.printSchema()

print(f"\n🔍 Amostra (3 primeiras linhas):")
display(transformed_df.limit(3))

# Verificar conversões
print(f"\n🔍 Verificação de conversões:")
valid_customer_id = transformed_df.filter(F.col("customer_id").isNotNull()).count()
valid_datetime = transformed_df.filter(F.col("order_datetime").isNotNull()).count()
valid_line_items = transformed_df.filter(F.col("number_of_line_items").isNotNull()).count()

print(f"   customer_id válidos: {valid_customer_id:,} ({(valid_customer_id/initial_count)*100:.2f}%)")
print(f"   order_datetime válidos: {valid_datetime:,} ({(valid_datetime/initial_count)*100:.2f}%)")
print(f"   number_of_line_items válidos: {valid_line_items:,} ({(valid_line_items/initial_count)*100:.2f}%)")

## 🗑️ Remoção de Duplicados

**IMPORTANTE**: Tabelas FATO **NÃO usam SCD Type 2**.

Tabelas fato representam **eventos imutáveis** (pedidos já realizados).
Não há "versões" de um pedido - cada order_number deve ser único.

**Estratégia**:
- Manter apenas o registro mais recente por order_number (ordenar por order_datetime DESC)
- Remover duplicados completos

In [0]:
print("🗑️ Removendo duplicados por order_number...")

# Window para ordenar por order_datetime DESC dentro de cada order_number
window_spec = Window.partitionBy("order_number").orderBy(F.col("order_datetime").desc_nulls_last())

# Adicionar row_number
ranked_df = transformed_df.withColumn("row_num", F.row_number().over(window_spec))

# Manter apenas o registro mais recente (row_num = 1)
deduped_df = ranked_df.filter(F.col("row_num") == 1).drop("row_num")

deduped_count = deduped_df.count()
removed_count = initial_count - deduped_count

print(f"✅ Duplicados removidos!")
print(f"\n📊 Estatísticas:")
print(f"   Registros antes: {initial_count:,}")
print(f"   Registros depois: {deduped_count:,}")
print(f"   Duplicados removidos: {removed_count:,}")
print(f"   Redução: {(removed_count/initial_count)*100:.2f}%")

# Verificar unicidade de order_number
unique_orders = deduped_df.select("order_number").distinct().count()
if unique_orders == deduped_count:
    print(f"\n✅ SUCESSO: Cada order_number é único na Silver! ({unique_orders:,} únicos)")
else:
    print(f"\n⚠️ ALERTA: Ainda existem duplicados! Únicos: {unique_orders:,}, Total: {deduped_count:,}")

## ✅ Validações Finais

Verificações de qualidade:
1. **Unicidade**: order_number deve ser único (PK)
2. **FK**: customer_id existe em silver.customers
3. **Arrays**: ordered_products não vazio (pedido sem produtos é inválido)
4. **Timestamps**: order_datetime não nulo
5. **Integridade**: number_of_line_items ≥ 1

In [0]:
print("✅ VALIDAÇÕES FINAIS")
print("=" * 80)

# 1. Unicidade de order_number
print("\n📊 1. Verificando unicidade (order_number):")
dup_check = deduped_df.groupBy("order_number").count().filter(F.col("count") > 1).count()
if dup_check == 0:
    print(f"   ✅ OK - Todos os order_numbers são únicos")
else:
    print(f"   ❌ ERRO - {dup_check:,} order_numbers ainda duplicados")

# 2. Validação de FK (customer_id existe em customers)
print("\n📊 2. Validando FK (customer_id → customers):")
try:
    customers_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers")
    valid_customer_ids = customers_df.select("CUSTOMER_ID").distinct()
    
    # Left anti join: encontrar customer_ids que NÃO existem em customers
    invalid_fk = deduped_df.filter(F.col("customer_id").isNotNull()) \
        .select("customer_id").distinct() \
        .join(valid_customer_ids, deduped_df["customer_id"] == valid_customer_ids["CUSTOMER_ID"], "left_anti")
    
    invalid_fk_count = invalid_fk.count()
    total_orders_with_customer = deduped_df.filter(F.col("customer_id").isNotNull()).count()
    
    if invalid_fk_count == 0:
        print(f"   ✅ OK - Todos os customer_ids existem em customers")
    else:
        print(f"   ⚠️ WARNING - {invalid_fk_count:,} customer_ids NÃO encontrados em customers")
        print(f"   📋 Exemplos de customer_ids inválidos:")
        display(invalid_fk.limit(10))
except Exception as e:
    print(f"   ⚠️ Não foi possível validar FK: {e}")
    print(f"   💡 Certifique-se que a tabela silver.customers existe")

# 3. Validação de ordered_products não vazio
print("\n📊 3. Validando ordered_products (não vazio):")
empty_products = deduped_df.filter(
    (F.col("ordered_products").isNull()) | (F.size(F.col("ordered_products")) == 0)
).count()
if empty_products == 0:
    print(f"   ✅ OK - Todos os pedidos têm produtos")
else:
    print(f"   ❌ ERRO - {empty_products:,} pedidos SEM produtos (inválido)")

# 4. Validação de order_datetime não nulo
print("\n📊 4. Validando order_datetime (não nulo):")
null_datetime = deduped_df.filter(F.col("order_datetime").isNull()).count()
if null_datetime == 0:
    print(f"   ✅ OK - Todos os pedidos têm order_datetime")
else:
    print(f"   ⚠️ WARNING - {null_datetime:,} pedidos sem order_datetime")

# 5. Validação de number_of_line_items ≥ 1
print("\n📊 5. Validando number_of_line_items ≥ 1:")
invalid_line_items = deduped_df.filter(
    (F.col("number_of_line_items").isNull()) | (F.col("number_of_line_items") < 1)
).count()
if invalid_line_items == 0:
    print(f"   ✅ OK - Todos os pedidos têm line_items ≥ 1")
else:
    print(f"   ⚠️ WARNING - {invalid_line_items:,} pedidos com line_items inválido")

print("\n✅ Validações concluídas!")

## 📊 Adicionar Colunas de Auditoria

Adicionando metadados para rastreabilidade:
- **data_quality_score**: Percentual de campos não-nulos
- **processed_at**: Timestamp do processamento
- **source_table**: Tabela de origem
- **pipeline_run_id**: ID único desta execução

In [0]:
print("📊 Adicionando colunas de auditoria...")

# Campos para cálculo de data_quality_score (8 campos principais)
fields_to_check = [
    "order_number",
    "customer_id",
    "customer_name",
    "order_datetime",
    "number_of_line_items",
    "clicked_items",
    "ordered_products",
    "promo_info"
]

total_fields = len(fields_to_check)

# Calcular data_quality_score
quality_expr = sum([
    F.when(F.col(field).isNotNull(), 1).otherwise(0) for field in fields_to_check
])

final_df = deduped_df.withColumn(
    "data_quality_score",
    (quality_expr / total_fields * 100).cast("int")
).withColumn(
    "processed_at",
    F.lit(processing_timestamp)
).withColumn(
    "source_table",
    F.lit(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
).withColumn(
    "pipeline_run_id",
    F.lit(pipeline_run_id)
)

print(f"✅ Colunas de auditoria adicionadas!")
print(f"\n📊 Schema final (antes da padronização):")
final_df.printSchema()

print(f"\n📊 Distribuição de data_quality_score:")
display(
    final_df.groupBy("data_quality_score")
    .count()
    .orderBy(F.col("data_quality_score").desc())
)

print(f"\n🔍 Amostra (3 primeiras linhas):")
display(final_df.limit(3))

In [0]:
# 🔤 PADRONIZAÇÃO: Converter TODOS os nomes de colunas para UPPER
print("🔤 Padronizando nomes de colunas para UPPER...")
print("=" * 80)

print(f"\n📋 Colunas ANTES da padronização:")
print(final_df.columns)

final_df = final_df.select([F.col(c).alias(c.upper()) for c in final_df.columns])

print(f"\n✅ Todas as colunas convertidas para UPPER!")
print(f"\n📋 Colunas DEPOIS da padronização:")
print(final_df.columns)

print(f"\n📋 Schema final:")
final_df.printSchema()

print(f"\n🔍 Amostra (3 primeiras linhas):")
display(final_df.limit(3))

## 💾 Gravar na Camada Silver

Escrevendo dados transformados na tabela Silver:
- **Formato**: Delta Lake
- **Modo**: Overwrite com schema evolution
- **Change Data Feed**: Habilitado
- **Otimização**: OPTIMIZE + ANALYZE TABLE

In [0]:
silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"💾 Gravando na tabela Silver: {silver_table_name}")

# Escrever no formato Delta
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(silver_table_name)

print(f"\n✅ Dados gravados com sucesso!")

# Otimizar tabela
print(f"\n🛠️ Otimizando tabela...")
spark.sql(f"OPTIMIZE {silver_table_name}")
print(f"✅ OPTIMIZE concluído!")

# Atualizar estatísticas
print(f"\n📊 Atualizando estatísticas...")
spark.sql(f"ANALYZE TABLE {silver_table_name} COMPUTE STATISTICS")
print(f"✅ ANALYZE TABLE concluído!")

# Verificar tabela criada
print(f"\n🔍 Verificação da tabela Silver:")
silver_df = spark.table(silver_table_name)
silver_verification = silver_df.count()
print(f"   Total de registros: {silver_verification:,}")

if silver_verification == deduped_count:
    print(f"   ✅ Verificação bem-sucedida! Todos os {deduped_count:,} registros foram gravados.")
else:
    print(f"   ⚠️ Divergência! Esperado: {deduped_count:,}, Encontrado: {silver_verification:,}")

## 📈 Relatório Final de Transformação

Resumo completo do pipeline de transformação Bronze → Silver.

In [0]:
print("📈 RELATÓRIO FINAL - SALES ORDERS TRANSFORMATION")
print("=" * 80)

# Estatísticas gerais
silver_df = spark.table(silver_table_name)
total_records = silver_df.count()

# Recuperar processed_at do DataFrame
processing_timestamp_final = silver_df.select("PROCESSED_AT").first()[0]

# Qualidade dos dados
avg_quality = silver_df.agg(F.avg("DATA_QUALITY_SCORE")).first()[0]
min_quality = silver_df.agg(F.min("DATA_QUALITY_SCORE")).first()[0]
max_quality = silver_df.agg(F.max("DATA_QUALITY_SCORE")).first()[0]

# Análise de arrays
avg_line_items = silver_df.agg(F.avg("NUMBER_OF_LINE_ITEMS")).first()[0]
max_line_items = silver_df.agg(F.max("NUMBER_OF_LINE_ITEMS")).first()[0]

# Análise temporal
min_date = silver_df.agg(F.min("ORDER_DATETIME")).first()[0]
max_date = silver_df.agg(F.max("ORDER_DATETIME")).first()[0]

print(f"\n📊 MÉTRICAS DE PROCESSAMENTO")
print(f"   📥 Registros Bronze: {initial_count:,}")
print(f"   📤 Registros Silver: {total_records:,}")
print(f"   🗑️ Duplicados removidos: {initial_count - total_records:,}")
print(f"   📈 Taxa de retenção: {(total_records/initial_count)*100:.2f}%")

print(f"\n📊 QUALIDADE DOS DADOS")
print(f"   📈 Score médio de qualidade: {avg_quality:.2f}%")
print(f"   📉 Score mínimo: {min_quality:.2f}%")
print(f"   📊 Score máximo: {max_quality:.2f}%")

print(f"\n📊 ANÁLISE DE PEDIDOS")
print(f"   📦 Média de itens por pedido: {avg_line_items:.2f}")
print(f"   📦 Máximo de itens em um pedido: {max_line_items}")

print(f"\n📊 ANÁLISE TEMPORAL")
print(f"   📅 Pedido mais antigo: {min_date}")
print(f"   📅 Pedido mais recente: {max_date}")

print(f"\n🔍 AUDITORIA")
print(f"   🆔 Pipeline Run ID: {pipeline_run_id}")
print(f"   ⏰ Processed At: {processing_timestamp_final}")
print(f"   📂 Source Table: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
print(f"   📂 Target Table: {silver_table_name}")

print(f"\n" + "="*80)
print(f"✅ TRANSFORMAÇÃO CONCLUÍDA COM SUCESSO! 🎉")
print("="*80)

# Dashboard de análise de produtos
print(f"\n📊 Top 10 pedidos por número de itens:")
display(
    silver_df.select(
        "ORDER_NUMBER",
        "CUSTOMER_NAME",
        "NUMBER_OF_LINE_ITEMS",
        "ORDER_DATETIME",
        F.size("ORDERED_PRODUCTS").alias("PRODUCTS_COUNT")
    ).orderBy(F.col("NUMBER_OF_LINE_ITEMS").desc())
    .limit(10)
)

# Análise de promoções
print(f"\n📊 Análise de promoções:")
with_promo = silver_df.filter(F.size(F.col("PROMO_INFO")) > 0).count()
without_promo = silver_df.filter(F.size(F.col("PROMO_INFO")) == 0).count()
print(f"   🎁 Pedidos COM promoção: {with_promo:,} ({(with_promo/total_records)*100:.2f}%)")
print(f"   📦 Pedidos SEM promoção: {without_promo:,} ({(without_promo/total_records)*100:.2f}%)")

# Salvar summary
summary_data = {
    "pipeline_run_id": str(pipeline_run_id),
    "processed_at": str(processing_timestamp_final),
    "source_table": f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}",
    "target_table": str(silver_table_name),
    "bronze_count": str(initial_count),
    "silver_count": str(total_records),
    "duplicates_removed": str(initial_count - total_records),
    "retention_rate": str(round((total_records/initial_count)*100, 2)),
    "avg_quality_score": str(round(avg_quality, 2)),
    "avg_line_items": str(round(avg_line_items, 2)),
    "orders_with_promo": str(with_promo),
    "orders_without_promo": str(without_promo)
}

print(f"\n📋 Summary data disponível na variável 'summary_data' para integração com dashboards/alertas")

# 🔀 EXPANSÃO DE ARRAYS E STRUCTS

## 🎯 Objetivo
Expandir os arrays nested da tabela sales_orders em estruturas mais analíticas:

### 1️⃣ PROMO_INFO → Colunas Agregadas
Adicionar colunas agregadas na tabela principal

### 2️⃣ ORDERED_PRODUCTS → Tabela sales_order_items ⭐
Explodir array de produtos em nova tabela Silver

### 3️⃣ CLICKED_ITEMS → Tabela sales_order_clicks
Explodir array de clicks em nova tabela Silver

## 1️⃣ Expandir PROMO_INFO em Colunas Agregadas

Vamos adicionar colunas derivadas de PROMO_INFO na tabela principal sales_orders

In [0]:
print("1️⃣ Expandindo PROMO_INFO em colunas agregadas...")
print("=" * 80)

orders_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

orders_expanded_df = orders_df.withColumn(
    "HAS_PROMO",
    F.when(F.size(F.col("PROMO_INFO")) > 0, True).otherwise(False)
).withColumn(
    "PROMO_COUNT",
    F.size(F.col("PROMO_INFO"))
).withColumn(
    "TOTAL_PROMO_DISCOUNT",
    F.when(
        F.size(F.col("PROMO_INFO")) > 0,
        F.aggregate(
            F.col("PROMO_INFO"),
            F.lit(0.0),
            lambda acc, x: acc + F.coalesce(x.promo_disc, F.lit(0.0))
        )
    ).otherwise(0.0)
).withColumn(
    "PROMO_ITEMS_LIST",
    F.when(
        F.size(F.col("PROMO_INFO")) > 0,
        F.array_join(
            F.transform(
                F.col("PROMO_INFO"),
                lambda x: x.promo_item
            ),
            ", "
        )
    ).otherwise(None)
)

print(f"\n✅ Colunas adicionadas com sucesso!")
print(f"\n📊 Amostra de pedidos COM promoção:")
display(
    orders_expanded_df.filter(F.col("HAS_PROMO") == True)
    .select(
        "ORDER_NUMBER",
        "CUSTOMER_NAME",
        "PROMO_COUNT",
        "TOTAL_PROMO_DISCOUNT",
        "PROMO_ITEMS_LIST"
    )
    .limit(10)
)

## 2️⃣ Explodir ORDERED_PRODUCTS → Tabela sales_order_items ⭐

Criar nova tabela Silver com uma linha por produto em cada pedido

In [0]:
print("2️⃣ Explodindo ORDERED_PRODUCTS → sales_order_items...")
print("=" * 80)

orders_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

items_df = orders_df.select(
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    "NUMBER_OF_LINE_ITEMS",
    F.explode("ORDERED_PRODUCTS").alias("product"),
    "DATA_QUALITY_SCORE",
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
).select(
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    "NUMBER_OF_LINE_ITEMS",
    F.col("product.curr").alias("CURRENCY"),
    F.col("product.id").alias("PRODUCT_ID"),
    F.col("product.name").alias("PRODUCT_NAME"),
    F.col("product.price").alias("PRODUCT_PRICE"),
    F.col("product.qty").alias("QUANTITY"),
    F.col("product.unit").alias("UNIT"),
    F.col("product.promotion_info.promo_disc").alias("PROMO_DISCOUNT"),
    F.col("product.promotion_info.promo_id").alias("PROMO_ID"),
    F.col("product.promotion_info.promo_item").alias("PROMO_ITEM"),
    F.col("product.promotion_info.promo_qty").alias("PROMO_QTY"),
    "DATA_QUALITY_SCORE",
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
)

items_df = items_df.withColumn(
    "ITEM_TOTAL",
    F.coalesce(F.col("PRODUCT_PRICE"), F.lit(0)) * F.coalesce(F.col("QUANTITY"), F.lit(0))
).withColumn(
    "HAS_ITEM_PROMO",
    F.when(F.col("PROMO_ID").isNotNull(), True).otherwise(False)
).withColumn(
    "ITEM_POSITION",
    F.row_number().over(Window.partitionBy("ORDER_NUMBER").orderBy(F.monotonically_increasing_id()))
)

total_items = items_df.count()
total_orders = items_df.select("ORDER_NUMBER").distinct().count()

print(f"\n✅ ORDERED_PRODUCTS explodido com sucesso!")
print(f"   Total de itens: {total_items:,}")
print(f"   Total de pedidos: {total_orders:,}")
print(f"   Média de itens por pedido: {total_items/total_orders:.2f}")

print(f"\n📊 Amostra (10 primeiros itens):")
display(items_df.limit(10))

items_table_name = f"{CATALOG}.{SILVER_SCHEMA}.sales_order_items"

print(f"\n💾 Gravando tabela: {items_table_name}")
items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(items_table_name)

print(f"✅ Tabela criada com sucesso!")
spark.sql(f"OPTIMIZE {items_table_name}")
spark.sql(f"ANALYZE TABLE {items_table_name} COMPUTE STATISTICS")
print(f"✅ Otimização concluída!")

## 3️⃣ Explodir CLICKED_ITEMS → Tabela sales_order_clicks

Criar nova tabela Silver com uma linha por click

In [0]:
print("3️⃣ Explodindo CLICKED_ITEMS → sales_order_clicks...")
print("=" * 80)

orders_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

clicks_df = orders_df.select(
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    F.explode("CLICKED_ITEMS").alias("click"),
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
).select(
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    F.col("click")[0].alias("CLICKED_PRODUCT_ID"),
    F.col("click")[1].cast("int").alias("CLICK_SCORE"),
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
)

clicks_df = clicks_df.withColumn(
    "CLICK_POSITION",
    F.row_number().over(Window.partitionBy("ORDER_NUMBER").orderBy(F.monotonically_increasing_id()))
)

total_clicks = clicks_df.count()
total_orders_with_clicks = clicks_df.select("ORDER_NUMBER").distinct().count()

print(f"\n✅ CLICKED_ITEMS explodido com sucesso!")
print(f"   Total de clicks: {total_clicks:,}")
print(f"   Total de pedidos com clicks: {total_orders_with_clicks:,}")
print(f"   Média de clicks por pedido: {total_clicks/total_orders_with_clicks:.2f}")

print(f"\n📊 Amostra (10 primeiros clicks):")
display(clicks_df.limit(10))

clicks_table_name = f"{CATALOG}.{SILVER_SCHEMA}.sales_order_clicks"

print(f"\n💾 Gravando tabela: {clicks_table_name}")
clicks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(clicks_table_name)

print(f"✅ Tabela criada com sucesso!")
spark.sql(f"OPTIMIZE {clicks_table_name}")
spark.sql(f"ANALYZE TABLE {clicks_table_name} COMPUTE STATISTICS")
print(f"✅ Otimização concluída!")

## 📊 Resumo Final - Expansão de Arrays

### ✅ Tabelas Criadas

| Tabela | Granularidade | Descrição |
|--------|---------------|----------|
| **sales_orders** | 1 linha = 1 pedido | Tabela original com arrays nested |
| **sales_order_items** ⭐ | 1 linha = 1 item | Produtos de cada pedido |
| **sales_order_clicks** | 1 linha = 1 click | Clicks antes da compra |

### 🔗 Relacionamentos
```
sales_orders (1) ──< (N) sales_order_items
     │                         │
     │                         └─ PRODUCT_ID → products
     │
     └──< (N) sales_order_clicks
```

In [0]:
print("📊 VERIFICAÇÃO FINAL - TODAS AS TABELAS SILVER")
print("=" * 80)

tables = [
    f"{CATALOG}.{SILVER_SCHEMA}.sales_orders",
    f"{CATALOG}.{SILVER_SCHEMA}.sales_order_items",
    f"{CATALOG}.{SILVER_SCHEMA}.sales_order_clicks"
]

print(f"\n📋 Tabelas Silver - Sales Orders:")
for table in tables:
    try:
        df = spark.table(table)
        count = df.count()
        cols = len(df.columns)
        print(f"\n✅ {table}")
        print(f"   Registros: {count:,}")
        print(f"   Colunas: {cols}")
    except Exception as e:
        print(f"\n❌ {table}")
        print(f"   Erro: {e}")

print(f"\n" + "="*80)
print(f"✅ EXPANSÃO DE ARRAYS CONCLUÍDA! 🎉")
print("="*80)